## Objective and business rationale
The goal is to train a regression model that predicts `ltv_revenue` for converted leads. This notebook focuses on monetizing high-value segments identified in the data forensic analysis, such as `watches` and `health_beauty`, where a single converted lead can generate outsized revenue.

The business rationale is to prioritize expected value: target leads that may generate the highest revenue, not just those with the highest conversion probability.

## 1. Data loading and preparation
We use the preprocessed Parquet dataset so the notebook remains fast and reproducible. This step focuses on loading the data safely and converting it to a pandas DataFrame.

In [1]:
import pandas as pd
import joblib
import pyarrow.parquet as pq
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
print("⏳ Loading the local Parquet backup...")
table = pq.read_table('C:\\Users\\User\\Desktop\\Software y Clases\\BigData\\OList\\olist-project-sa\\Data\\Processed\\abt_marketing.parquet')

# Remove Arrow/Pandas metadata that can cause conversion issues
# This avoids problems with metadata fields such as 'dbdate' during DataFrame creation
safe_table = table.replace_schema_metadata(None)

df_rf = safe_table.to_pandas()
print(f"✅ Data loaded successfully: {df_rf.shape}")

⏳ Loading the local Parquet backup...
✅ Data loaded successfully: (8000, 11)


## 2. Training on actual converted leads only
We keep only converted leads so the revenue regressor learns from real outcomes. This prevents the model from being influenced by leads that never generated revenue.

In [3]:
# Keep only leads that converted, so the regressor learns from actual revenue outcomes
# This is critical because the target variable represents realized revenue, not potential revenue.
df_reg = df_rf[df_rf['converted'] == 1].copy()

In [4]:
# Feature engineering based on data forensic findings
high_value_list = ['watches', 'health_beauty', 'audio_video_electronics']
df_reg['is_high_value_segment'] = df_reg['business_segment'].isin(high_value_list).astype(int)

features = ['origin', 'lead_type', 'is_high_value_segment']
X = pd.get_dummies(df_reg[features], drop_first=True)
y = df_reg['ltv_revenue']  # our economic target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training LTV regressor with {X_train.shape[0]} converted leads...")

Training LTV regressor with 673 converted leads...


## 4. Model selection and evaluation
Random Forest Regressor is chosen for its robustness to outliers and ability to model non-linear relationships. We evaluate with MAE because it provides a direct dollar error metric that stakeholders can interpret.

In [5]:
# Use Random Forest Regressor for robustness against outliers and non-linear relationships
regressor = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
regressor.fit(X_train, y_train)

# Evaluation
y_pred = regressor.predict(X_test)
print(f"📊 MAE (Mean Absolute Error): ${mean_absolute_error(y_test, y_pred):.2f}")
print(f"📊 R2 score: {r2_score(y_test, y_pred):.4f}")

# Save artifacts for Streamlit
joblib.dump(regressor, r'C:\Users\User\Desktop\Software y Clases\BigData\OList\olist-project-sa\Model\ltv_regressor_model.joblib')
joblib.dump(X_train.columns.tolist(), r'C:\Users\User\Desktop\Software y Clases\BigData\OList\olist-project-sa\Model\regressor_features.joblib')
print("✅ Revenue regressor and feature list saved successfully.")

📊 MAE (Mean Absolute Error): $1966.45
📊 R2 score: -0.1490
✅ Revenue regressor and feature list saved successfully.


## 5. Final insights
- The model is trained only on converted leads, so its predictions are best interpreted as revenue forecasts for leads that have already been qualified.
- High-value segments such as `watches` and `health_beauty` are important because they reflect actual cases with much larger revenue outcomes.
- This model is not a conversion probability model; it is a revenue estimation model for converted or high-quality leads.
- MAE is reported in dollars, which makes the average error directly interpretable for business stakeholders and helps set realistic expectations.